In [1]:
import cv2
print(cv2.__version__)

4.8.0


In [2]:
import os
import time
import cv2
import numpy as np
from model.yolo_model import YOLO

In [3]:
def process_image(img):
    """Resize, reduce and expand image.

    # Argument:
        img: original image.

    # Returns
        image: ndarray(64, 64, 3), processed image.
    """
    image = cv2.resize(img, (416, 416),
                       interpolation=cv2.INTER_CUBIC)
    image = np.array(image, dtype='float32')
    image /= 255.
    image = np.expand_dims(image, axis=0)

    return image

In [4]:
def get_classes(file):
    """Get classes name.

    # Argument:
        file: classes name for database.

    # Returns
        class_names: List, classes name.

    """
    with open(file) as f:
        class_names = f.readlines()
    class_names = [c.strip() for c in class_names]

    return class_names

In [5]:
def draw(image, boxes, scores, classes, all_classes):
    """Draw the boxes on the image.

    # Argument:
        image: original image.
        boxes: ndarray, boxes of objects.
        classes: ndarray, classes of objects.
        scores: ndarray, scores of objects.
        all_classes: all classes name.
    """
    for box, score, cl in zip(boxes, scores, classes):
        x, y, w, h = box

        top = max(0, np.floor(x + 0.5).astype(int))
        left = max(0, np.floor(y + 0.5).astype(int))
        right = min(image.shape[1], np.floor(x + w + 0.5).astype(int))
        bottom = min(image.shape[0], np.floor(y + h + 0.5).astype(int))

        cv2.rectangle(image, (top, left), (right, bottom), (255, 0, 0), 2)
        cv2.putText(image, '{0} {1:.2f}'.format(all_classes[cl], score),
                    (top, left - 6),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 0, 255), 1,
                    cv2.LINE_AA)

        print('class: {0}, score: {1:.2f}'.format(all_classes[cl], score))
        print('box coordinate x,y,w,h: {0}'.format(box))

    print()

In [6]:
def detect_image(image, yolo, all_classes):
    """Use yolo v3 to detect images.

    # Argument:
        image: original image.
        yolo: YOLO, yolo model.
        all_classes: all classes name.

    # Returns:
        image: processed image.
    """
    pimage = process_image(image)

    start = time.time()
    boxes, classes, scores = yolo.predict(pimage, image.shape)
    end = time.time()

    print('time: {0:.2f}s'.format(end - start))

    if boxes is not None:
        draw(image, boxes, scores, classes, all_classes)

    return image

In [7]:
def detect_video(video, yolo, all_classes):
    """Use yolo v3 to detect video.

    # Argument:
        video: video file.
        yolo: YOLO, yolo model.
        all_classes: all classes name.
    """
    video_path = os.path.join("videos", "test", video)
    if not os.path.exists(video_path):
        os.makedirs(video_path)
    camera = cv2.VideoCapture(video_path)
    
    if not camera.isOpened():
        print("Failed to open input video!")
        return
    
    cv2.namedWindow("detection", cv2.WINDOW_AUTOSIZE)

    # Prepare for saving the detected video
    sz = (int(camera.get(cv2.CAP_PROP_FRAME_WIDTH)),
        int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')

    
    vout = cv2.VideoWriter()
    vout.open(os.path.join("videos", "res", video), fourcc, 10, sz, True)

    if vout.isOpened():
        print("Video writer is open!")
    else:
        print("Failed to open video writer!")

    while True:
        res, frame = camera.read()

        if not res:
            break

        image = detect_image(frame, yolo, all_classes)
        cv2.imshow("detection", image)

        # Save the video frame by frame
        vout.write(image)

        if cv2.waitKey(110) & 0xff == 27:
                break

    vout.release()
    camera.release()
    

In [8]:
yolo = YOLO(0.3, 0.5)
file = 'data/coco_classes.txt'
all_classes = get_classes(file)

In [9]:
# Detectar imagenes

In [10]:
f = 'coche.jpg'
path = 'imagenes/'+f
image = cv2.imread(path)
image = detect_image(image, yolo, all_classes)
cv2.imwrite('imagenes/res/' + f, image)

1/1 [==============================] - 1s 575ms/step
time: 0.60s
class: person, score: 0.42
box coordinate x,y,w,h: [1459.3401432  1110.21429229  477.19963789  883.44673264]
class: car, score: 1.00
box coordinate x,y,w,h: [ 393.95588636 1231.32865167 1781.16252422  691.70623577]



True

In [11]:
detect_video('grabacion.mp4',yolo,all_classes)

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'


Video writer is open!
1/1 [==============================] - 0s 258ms/step
time: 0.28s
class: person, score: 0.99
box coordinate x,y,w,h: [267.27832794   2.36285448 609.91493225 710.43936253]

1/1 [==============================] - 0s 250ms/step
time: 0.27s
class: person, score: 0.99
box coordinate x,y,w,h: [267.32126236   3.08557034 608.3297348  708.9463377 ]

1/1 [==============================] - 0s 250ms/step
time: 0.28s
class: person, score: 0.99
box coordinate x,y,w,h: [264.60390091   4.18806553 608.16493988 705.23506165]

1/1 [==============================] - 0s 249ms/step
time: 0.28s
class: person, score: 0.99
box coordinate x,y,w,h: [260.24196625   6.89342737 614.91661072 701.4200592 ]

1/1 [==============================] - 0s 246ms/step
time: 0.27s
class: person, score: 0.99
box coordinate x,y,w,h: [266.94984436   3.65876913 604.79179382 708.31290722]

1/1 [==============================] - 0s 248ms/step
time: 0.27s
class: person, score: 0.99
box coordinate x,y,w,h: [270.10

1/1 [==============================] - 0s 255ms/step
time: 0.28s
class: person, score: 1.00
box coordinate x,y,w,h: [165.2338028   19.15753841 729.22149658 672.21106052]
class: cell phone, score: 0.85
box coordinate x,y,w,h: [ 65.50748825  32.35866308 310.81729889 350.32696724]

1/1 [==============================] - 0s 254ms/step
time: 0.28s
class: person, score: 1.00
box coordinate x,y,w,h: [169.00657654  17.29958296 727.48153687 675.97259045]
class: cell phone, score: 0.87
box coordinate x,y,w,h: [ 65.88547707  30.70024252 302.39614487 354.01451111]

1/1 [==============================] - 0s 253ms/step
time: 0.28s
class: person, score: 1.00
box coordinate x,y,w,h: [168.92578125  17.62700558 727.4356842  676.70498371]
class: cell phone, score: 0.86
box coordinate x,y,w,h: [ 53.74456406  60.38378835 324.35718536 333.96585703]

1/1 [==============================] - 0s 249ms/step
time: 0.28s
class: person, score: 1.00
box coordinate x,y,w,h: [162.76424408  16.90761566 738.04191589 677.

1/1 [==============================] - 0s 250ms/step
time: 0.28s
class: person, score: 1.00
box coordinate x,y,w,h: [164.75822449  34.10284996 716.70272827 649.14341927]
class: cell phone, score: 0.83
box coordinate x,y,w,h: [179.00230408  94.50958729 265.60642242 347.43206978]

1/1 [==============================] - 0s 280ms/step
time: 0.31s
class: person, score: 1.00
box coordinate x,y,w,h: [190.68977356  33.89434576 682.06542969 646.42082691]
class: remote, score: 0.85
box coordinate x,y,w,h: [177.9990387  128.4454751  260.1912117  350.09046078]

1/1 [==============================] - 0s 256ms/step
time: 0.28s
class: person, score: 1.00
box coordinate x,y,w,h: [177.4048996   36.54320955 691.9102478  644.93299484]
class: cell phone, score: 0.84
box coordinate x,y,w,h: [220.95317841 155.30174732 220.93545914 335.87664127]

1/1 [==============================] - 0s 258ms/step
time: 0.28s
class: person, score: 1.00
box coordinate x,y,w,h: [200.46203613  35.1892519  666.50588989 647.3632